# Тема 6. Кластеризация и снижение размерности

**Практическая работа.** Основные данные — Wine. `target` не используется до задания 10.  
Заполните все блоки **TODO**, сохраните таблицы/графики и добавьте короткий вывод после каждого задания.

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (silhouette_score, calinski_harabasz_score,
    davies_bouldin_score, adjusted_rand_score, normalized_mutual_info_score)
RANDOM_STATE=42
print(sys.version)
print('sklearn', sklearn.__version__)

In [ ]:
wine=load_wine(as_frame=True)
X=wine.data.copy()
y_hidden=wine.target.copy()  # НЕ ИСПОЛЬЗОВАТЬ ДО ЗАДАНИЯ 10
X.shape, X.head()

## 1. Аудит и постановка

**Цель:** Сформулировать объект анализа, прикладную цель и риски.  
**Задание:** Вывести shape, типы, пропуски, описательные статистики, дубликаты. Составить data card: строка, признак, период, предполагаемое применение кластеров. Не анализировать target.  
**Ожидаемый результат:** Таблица аудита и 5–8 предложений постановки.

In [ ]:
# TODO 1
X.info()
summary = X.describe().T
summary['missing'] = X.isna().sum()
summary['n_unique'] = X.nunique()
summary
# Добавьте расчеты, визуализации и проверки.

**Вывод:** TODO — 3–6 содержательных предложений.

## 2. Масштабирование

**Цель:** Показать, как единицы измерения меняют расстояния.  
**Задание:** Сравнить распределения признаков и расстояния между объектами до и после StandardScaler. Объяснить, какие признаки доминируют без scaling. Дополнительно сравнить RobustScaler.  
**Ожидаемый результат:** Таблица квантилей расстояний и аргументированный выбор scaler.

In [ ]:
# TODO 2
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import pairwise_distances
Xs = StandardScaler().fit_transform(X)
D_raw = pairwise_distances(X)
D_std = pairwise_distances(Xs)
# Добавьте расчеты, визуализации и проверки.

**Вывод:** TODO — 3–6 содержательных предложений.

## 3. K-means и диапазон K

**Цель:** Исследовать зависимость качества от числа кластеров.  
**Задание:** Для K=2…10 вычислить inertia, silhouette, Calinski–Harabasz и Davies–Bouldin. Построить графики. Не выбирать K только по одной метрике.  
**Ожидаемый результат:** Таблица метрик, 2–4 кандидата K и объяснение компромисса.

In [ ]:
# TODO 3
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
rows=[]
for k in range(2, 11):
    labels=KMeans(n_clusters=k,n_init=30,random_state=42).fit_predict(Xs)
    rows.append([k, silhouette_score(Xs, labels),
                 calinski_harabasz_score(Xs, labels),
                 davies_bouldin_score(Xs, labels)])
# Добавьте расчеты, визуализации и проверки.

**Вывод:** TODO — 3–6 содержательных предложений.

## 4. Устойчивость K-means

**Цель:** Отделить удачный запуск от воспроизводимой структуры.  
**Задание:** Для выбранных K выполнить 30 запусков с разными random_state. Сравнить inertia, silhouette и попарный ARI между разбиениями. Затем выполнить bootstrap-оценку устойчивости.  
**Ожидаемый результат:** Распределения метрик и вывод о наиболее устойчивом K.

In [ ]:
# TODO 4
from sklearn.metrics import adjusted_rand_score
labelings=[]
for seed in range(30):
    labelings.append(KMeans(n_clusters=3,n_init=1,random_state=seed).fit_predict(Xs))
ari=[adjusted_rand_score(labelings[i], labelings[j])
     for i in range(len(labelings)) for j in range(i)]
# Добавьте расчеты, визуализации и проверки.

**Вывод:** TODO — 3–6 содержательных предложений.

## 5. Профили кластеров

**Цель:** Перевести геометрическое разбиение в предметное описание.  
**Задание:** Для выбранной модели вычислить размеры, медианы, IQR и standardized lift признаков. Найти типичный объект, ближайший к центроиду, и три пограничных объекта.  
**Ожидаемый результат:** Профиль каждого кластера и осторожное описательное название.

In [ ]:
# TODO 5
labels = KMeans(n_clusters=3,n_init=50,random_state=42).fit_predict(Xs)
profile = X.assign(cluster=labels).groupby('cluster').median()
profile_z = (profile - X.mean()) / X.std()
profile_z.T
# Добавьте расчеты, визуализации и проверки.

**Вывод:** TODO — 3–6 содержательных предложений.

## 6. Иерархическая модель

**Цель:** Исследовать альтернативную геометрическую гипотезу.  
**Задание:** Сравнить Ward, complete, average и single linkage. Построить дендрограмму. Выбрать уровень разреза и сравнить разбиение с K-means через ARI.  
**Ожидаемый результат:** Дендрограмма, таблица сравнений и объяснение эффекта linkage.

In [ ]:
# TODO 6
from sklearn.cluster import AgglomerativeClustering
for linkage in ['ward','complete','average','single']:
    labels_h = AgglomerativeClustering(n_clusters=3, linkage=linkage).fit_predict(Xs)
    print(linkage, silhouette_score(Xs, labels_h))
# Добавьте расчеты, визуализации и проверки.

**Вывод:** TODO — 3–6 содержательных предложений.

## 7. DBSCAN/HDBSCAN

**Цель:** Изучить плотностную кластеризацию и понятие шума.  
**Задание:** Построить k-distance plot. Исследовать сетку eps и min_samples для DBSCAN. Зафиксировать число кластеров, долю шума и silhouette по не-шумовым точкам. Дополнительно применить HDBSCAN при доступной версии.  
**Ожидаемый результат:** Карта параметров, объяснение выбранного режима и природы шума.

In [ ]:
# TODO 7
from sklearn.cluster import DBSCAN
for eps in [0.8, 1.0, 1.2, 1.5, 2.0]:
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(Xs)
    n = len(set(lab)) - (1 if -1 in lab else 0)
    print(eps, n, (lab == -1).mean())
# Добавьте расчеты, визуализации и проверки.

**Вывод:** TODO — 3–6 содержательных предложений.

## 8. Gaussian Mixture

**Цель:** Получить мягкие назначения и исследовать неопределенность.  
**Задание:** Сравнить covariance_type и K по BIC/AIC. Для выбранной модели проанализировать max probability и entropy. Найти наиболее неоднозначные объекты.  
**Ожидаемый результат:** Графики BIC/AIC, таблица неоднозначных наблюдений.

In [ ]:
# TODO 8
from sklearn.mixture import GaussianMixture
models=[]
for cov in ['spherical','diag','tied','full']:
    for k in range(2,7):
        gm=GaussianMixture(k,covariance_type=cov,n_init=10,random_state=42).fit(Xs)
        models.append((cov,k,gm.bic(Xs),gm.aic(Xs)))
# Добавьте расчеты, визуализации и проверки.

**Вывод:** TODO — 3–6 содержательных предложений.

## 9. PCA

**Цель:** Освоить линейное снижение размерности и интерпретацию компонент.  
**Задание:** Обучить PCA на scaled data. Построить scree и cumulative variance. Найти минимальное число компонент для 80%, 90% и 95%. Проанализировать loadings первых трех компонент.  
**Ожидаемый результат:** Графики, таблица loadings и содержательное описание осей.

In [ ]:
# TODO 9
from sklearn.decomposition import PCA
pca=PCA().fit(Xs)
cum=pca.explained_variance_ratio_.cumsum()
loadings=pd.DataFrame(pca.components_.T,index=X.columns,
                      columns=[f'PC{i+1}' for i in range(X.shape[1])])
# Добавьте расчеты, визуализации и проверки.

**Вывод:** TODO — 3–6 содержательных предложений.

## 10. Внешняя диагностика

**Цель:** Понять разницу между структурой и известными классами.  
**Задание:** Только теперь открыть y_hidden. Вычислить ARI и NMI для выбранных моделей. Построить contingency table. Объяснить, почему ARI не использовался для unsupervised-настройки.  
**Ожидаемый результат:** Внешние метрики и интерпретация расхождений.

In [ ]:
# TODO 10
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import pandas as pd
print(adjusted_rand_score(y_hidden, labels))
print(pd.crosstab(y_hidden, labels, normalize='index'))
# Добавьте расчеты, визуализации и проверки.

**Вывод:** TODO — 3–6 содержательных предложений.

## 11. PCA против нелинейной карты

**Цель:** Научиться критически читать 2D-embedding.  
**Задание:** Сравнить PCA(2), t-SNE и, при доступности, UMAP. Для t-SNE проверить не менее трех perplexity и двух seed. Цветом показать одни и те же cluster labels. Не кластеризовать только по красивой картинке.  
**Ожидаемый результат:** Набор карт и список устойчивых/неустойчивых визуальных наблюдений.

In [ ]:
# TODO 11
from sklearn.manifold import TSNE
Z_tsne = TSNE(n_components=2, perplexity=30, init='pca',
              learning_rate='auto', random_state=42).fit_transform(Xs)
# Добавьте расчеты, визуализации и проверки.

**Вывод:** TODO — 3–6 содержательных предложений.

## 12. Итоговый model memo

**Цель:** Сформулировать решение, пригодное для обсуждения с заказчиком.  
**Задание:** Выбрать финальный pipeline. Описать цель, preprocessing, алгоритм, параметры, метрики, профили, устойчивость, правило для новых объектов, ограничения и мониторинг. Предложить один эксперимент для проверки полезности сегментов.  
**Ожидаемый результат:** Model memo до 2 страниц и итоговая таблица решений.

In [ ]:
# TODO 12
# Минимум для воспроизводимости
import sklearn, numpy, pandas
print(sklearn.__version__, numpy.__version__, pandas.__version__)
# Добавьте расчеты, визуализации и проверки.

**Вывод:** TODO — 3–6 содержательных предложений.

## Итоговый model memo

1. Прикладная цель и единица анализа.  
2. Признаки, preprocessing и метрика.  
3. Сравненные модели и критерии.  
4. Выбранное решение и профили кластеров.  
5. Устойчивость и внешняя диагностика.  
6. Правило для новых объектов.  
7. Ограничения, drift и эксперимент по проверке пользы.